# Concept

Once your model produces a raw prediction tensor (called logits), you must quantify exactly how wrong those predictions are compared to your ground-truth target data. This error metric is calculated by a Loss Function ($L$).
PyTorch packages these mathematical calculations into modular classes inside the torch.nn library. The two most common foundational loss functions used across machine learning pipelines are:
nn.MSELoss() (Mean Squared Error): Used for regression tasks (predicting continuous numeric values like housing prices or coordinates). It measures the average squared difference between predictions and actual targets:$$L_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^{n} (y_{\text{pred}} - y_{\text{true}})^2$$
nn.CrossEntropyLoss(): Used for classification tasks (predicting distinct categorical classes like "Cat" vs "Dog"). It converts raw model logits into probability distributions using a softmax function, and then computes the negative log-likelihood of the correct class.

In [1]:
import torch
import torch.nn as nn

# Mock model predictions (regression values) and actual targets
predictions = torch.tensor([2.5, 4.0, 1.1], requires_grad=True)
ground_truth = torch.tensor([3.0, 4.0, 1.0])

# 1. Instantiate the native loss criterion module object
criterion = nn.MSELoss()

# 2. Compute the loss value by evaluating predictions against targets
loss = criterion(predictions, ground_truth)

print(f"Calculated MSE Loss: {loss.item()}") 
# Output: 0.0867 -> ((2.5-3.0)^2 + (4.0-4.0)^2 + (1.1-1.0)^2) / 3

# The loss tensor tracks gradients back through your model automatically!
print(f"Loss Gradient Function: {loss.grad_fn}")
# Output: <MseLossBackward0 object>

Calculated MSE Loss: 0.08666666597127914
Loss Gradient Function: <MseLossBackward0 object at 0x7fbe4f8b6110>


# Exercise

Write a script that calculates the Mean Squared Error between a set of mock model outputs and actual sensor readings.

    Create a tensor named preds containing values [10.5, 20.0] and enable gradient tracking on it.

    Create a tensor named targets containing values [10.0, 22.0] (no gradients needed).

    Instantiate the native nn.MSELoss() module.

    Pass your parameters into the loss module, compute the total scalar error, and print its value. Then, trigger a backward pass on that loss tensor and print the calculated gradients stored inside preds.grad.

In [4]:
preds = torch.tensor([10.5, 20.0], requires_grad=True)
targets = torch.tensor([10.0, 22.0], requires_grad=True)

criterion = nn.MSELoss()
loss = criterion(preds, targets)

print("Loss:", loss.item())
loss.backward()
print("Gradients:", preds.grad)

Loss: 2.125
Gradients: tensor([ 0.5000, -2.0000])


# Concept

Now let's break down classification loss. In a multi-class classification problem (e.g., identifying a handwritten digit from 0 to 9), your model outputs a vector of raw, unnormalized values called logits.To calculate cross-entropy loss, PyTorch's nn.CrossEntropyLoss() handles a multi-step mathematical pipeline internally for maximum numerical stability:It applies the Softmax function to transform your raw unnormalized logits into an absolute probability distribution where all values sit between 0 and 1 and sum to 1.It calculates the Negative Log-Likelihood ($L = -\log(p_{\text{correct}})$), penalizing your model heavily if its confidence for the correct class is close to 0.Critical Layout Rule:nn.CrossEntropyLoss() expects two distinct input tensor shapes:Predictions: A 2D tensor of shape (batch_size, num_classes) containing raw float logits. Do not apply a manual softmax layer yourself before passing values to this loss function.Targets: A 1D tensor of shape (batch_size) containing integer class indices ($0, 1, 2, \dots$) representing the true correct label for each sample.

In [5]:
# Simulating a batch size of 2 samples across 3 possible classes (e.g., [Cat, Dog, Bird])
# Raw unnormalized model output logits
logits = torch.tensor([[1.5, 0.2, -0.8],   # Sample 1 logits
                       [-0.5, 3.1, 0.4]],  # Sample 2 logits
                      requires_grad=True)

# Ground-truth labels for the batch:
# Sample 1 is class index 0 (Cat), Sample 2 is class index 1 (Dog)
class_targets = torch.tensor([0, 1], dtype=torch.long) # Must be integers!

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, class_targets)

print(f"CrossEntropy Loss: {loss.item()}")

CrossEntropy Loss: 0.20358498394489288


# Exercise

Write a script that calculates the classification error for a batch of data.

    Create a raw logits tensor named logits of shape (3, 4) containing random floating-point values (use torch.randn(3, 4, requires_grad=True)). This simulates a batch of 3 samples evaluated across 4 target categories.

    Create an integer target tensor named labels containing values [2, 0, 3]. This states that sample 1 belongs to class 2, sample 2 to class 0, and sample 3 to class 3. Ensure it uses an integer data type (dtype=torch.long).

    Instantiate the native nn.CrossEntropyLoss() module.

    Compute the scalar loss and print it.

In [7]:
logits = torch.randn(3, 4, requires_grad=True)
target = torch.tensor([2, 0, 3], dtype=torch.long)

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

print(loss)

tensor(2.5636, grad_fn=<NllLossBackward0>)


# Concept

When calculating loss across a batch of multiple samples, PyTorch computes a separate error value for each individual sample. By default, it automatically compresses these individual errors down into a single scalar value by calculating their average. This behavior is controlled by the reduction parameter inside the loss function constructor.

The reduction parameter accepts three configurations:

    reduction='mean' (Default): Averages the individual sample losses into a single scalar.

    reduction='sum': Adds up all the individual losses into a single scalar.

    reduction='none': Disables compression entirely. It returns a raw tensor containing the exact calculated loss for each individual data sample in your batch. This is highly useful for debugging or weighting specific samples.

In [8]:
preds = torch.tensor([1.0, 5.0], requires_grad=True)
truth = torch.tensor([2.0, 5.0])

# Instantiate with reduction='none' to see individual errors
criterion_none = nn.MSELoss(reduction='none')
loss_tensor = criterion_none(preds, truth)

print(loss_tensor) 
# Output: tensor([1.0000, 0.0000]) -> ((1-2)^2 is 1, (5-5)^2 is 0)
# Notice it preserves the shape of the batch instead of collapsing it!

tensor([1., 0.], grad_fn=<MseLossBackward0>)
